# DuckDB GeoParquet Benchmarks

This notebook compares direct DuckDB queries across GeoParquet layouts: the
Microsoft Planetary Computer and Chris Holmes Sentinel-2 L2A datasets. The
Holmes source and modified Microsoft layouts are compared with stac-hash-sorted
layouts produced by
`cosgp convert`. It builds on the same `cosgp.cli.benchmark` framework that
backs the `cosgp benchmark run`/`compare` CLI commands, so results exported here
can be compared with `cosgp benchmark compare` directly.

For a single dataset with a fixed query suite, prefer the CLI:

```sh
uv run cosgp benchmark run my-dataset "optimized/*.parquet"
uv run cosgp benchmark compare benchmark-results/run-a-* benchmark-results/run-b-*
```

This notebook exists for deeper cross-layout comparisons that the CLI's
single-dataset, single-run model doesn't cover: running a fixed query suite
across several dataset variants at once, inspecting Parquet metadata, and
digging into `EXPLAIN ANALYZE` plans.

## Setup

Requires the `benchmark` extra and the `notebooks` dependency group:

```sh
uv sync --extra benchmark --group notebooks
uv run --group notebooks jupyter lab notebooks/duckdb-geoparquet-benchmarks.ipynb
```

In [ ]:
from __future__ import annotations

import importlib
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd

from cosgp.cli.benchmark import queries as benchmark_queries
from cosgp.cli.benchmark import runner as benchmark_runner

importlib.reload(benchmark_queries)
importlib.reload(benchmark_runner)

from cosgp.cli.benchmark.queries import (
    DEFAULT_REPEATS,
    QUERIES,
    BenchmarkSuite,
)
from cosgp.cli.benchmark.runner import (
    BenchmarkResult,
    BenchmarkRunner,
    queries_for_suite,
    runnable_queries,
    skipped_queries,
    sql_literal,
)

## Remote Datasets

All benchmark variants are read directly from Source Cooperative so object-store
listing, network latency, and HTTP range reads are measured consistently. Each
source is paired with hash-sorted variants produced by `cosgp convert`. The
converter hashes each item's datetime and bbox center over a year-long global
extent, partitions the ordered hash range into equal-item buckets, sorts each
bucket by `hash:hash`, and writes 150,000-row row groups. Holmes uses 2 GB,
500 MB, and 250 MB bucket targets, producing one, four, and eight files.

In [ ]:
BENCHMARK_PARAMS = {
    'microsoft': {
        'collection': 'sentinel-2-l2a',
        'id': 'S2B_MSIL2A_20250101T031029_R075_T52VCJ_20250101T050301',
        'start_datetime': datetime(2025, 6, 1, tzinfo=UTC),
        'end_datetime': datetime(2025, 7, 1, tzinfo=UTC),
        'minx': -109.0, 'miny': 37.0, 'maxx': -102.0, 'maxy': 41.0,
        'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
        'max_cloud_cover': 20.0,
    },
    'cholmes': {
        'collection': 'sentinel-2-l2a',
        'id': 'S2B_MSIL2A_20200115T180709_R041_T13TCF_20201002T191013',
        'start_datetime': datetime(2020, 1, 15, tzinfo=UTC),
        'end_datetime': datetime(2020, 1, 16, tzinfo=UTC),
        'minx': -109.0, 'miny': 37.0, 'maxx': -102.0, 'maxy': 41.0,
        'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
        'max_cloud_cover': 20.0,
    },
}

DATASET_FAMILIES = {
    'microsoft': 'microsoft',
    'hashed': 'microsoft',
    'hashed_matched_file_count': 'microsoft',
    'cholmes': 'cholmes',
    'cholmes_hashed': 'cholmes',
    'cholmes_hashed_4_files': 'cholmes',
    'cholmes_hashed_8_files': 'cholmes',
}

In [ ]:
SOURCE_COOPERATIVE = 's3://us-west-2.opendata.source.coop'
DEVELOPMENTSEED = f'{SOURCE_COOPERATIVE}/developmentseed/stac-geoparquet'
CHOLMES = f'{SOURCE_COOPERATIVE}/cholmes/stac-geoparquet-s2l2a'

In [ ]:
DATASETS = {
    'microsoft': f'{DEVELOPMENTSEED}/mspc-sentinel-2-l2a/*.parquet',
    'hashed': f'{DEVELOPMENTSEED}/mspc-sentinel-2-l2a-hashed/*.parquet',
    'hashed_matched_file_count': f'{DEVELOPMENTSEED}/mspc-sentinel-2-l2a-hashed-matched/*.parquet',
    'cholmes': f'{CHOLMES}/sentinel-2-l2a.parquet',
    'cholmes_hashed': f'{DEVELOPMENTSEED}/cholmes-sentinel-2-l2a-hashed/*.parquet',
    'cholmes_hashed_4_files': f'{DEVELOPMENTSEED}/cholmes-sentinel-2-l2a-hashed-4-files/*.parquet',
    'cholmes_hashed_8_files': f'{DEVELOPMENTSEED}/cholmes-sentinel-2-l2a-hashed-8-files/*.parquet',
}

DATASETS

## Dataset Variants

Variants within each family contain the same items, so query row counts should
match. Compare layouts only within their family: Microsoft and Holmes cover
different acquisition periods and have very different collection sizes.

In [ ]:
DATASETS

## Query Suite

`QUERIES` comes straight from `cosgp.cli.benchmark.queries`, so the notebook
and CLI use the same benchmark definitions. `BENCHMARK_SUITE` controls whether
the notebook runs isolated `core` queries, multi-predicate `composite` queries,
or both. This notebook runs `all` so the isolated diagnostics and realistic
composite STAC searches are measured together.

- Core queries isolate one filter or ordering dimension.
- Composite queries simulate realistic STAC searches with multiple predicates.
- Hash queries run only for datasets containing a `hash:hash` column.

Every unavailable query is reported with the missing columns or parameters.

In [ ]:
BENCHMARK_SUITE = BenchmarkSuite.all
SELECTED_QUERIES = queries_for_suite(QUERIES, BENCHMARK_SUITE)

## Helpers

`BenchmarkRunner` (from the CLI framework) owns the DuckDB connection, query
timing, and `run.json` export. This notebook owns the fixed benchmark scenario
in `BENCHMARK_PARAMS`; each dataset family has parameters matching its acquisition
dates. For hash-sorted datasets, untimed setup first runs the exact STAC
predicates and takes the minimum and maximum hash among those matches. This
conservative interval can admit false positives, which the exact predicates
remove, but it cannot exclude a matching item. This is a benchmark correctness
method, not yet a production query-planning strategy, because setup scans the
exact matches before timing begins.

In [ ]:
def dataset_columns(runner: BenchmarkRunner, path: str) -> set[str]:
    cursor = runner.connection.execute(
        f'DESCRIBE SELECT * FROM read_parquet({sql_literal(path)}) LIMIT 0'
    )
    return {row[0] for row in cursor.fetchall()}


def resolve_all_params(
    runner: BenchmarkRunner, dataset_name: str, path: str
) -> tuple[dict[str, object], set[str]]:
    family = DATASET_FAMILIES[dataset_name]
    params = dict(BENCHMARK_PARAMS[family])
    columns = dataset_columns(runner, path)
    if 'hash:hash' in columns:
        params.update(runner.resolve_hash_range(path, columns, params))
    return params, columns

## Parquet Metadata

Run this before timing queries. It verifies file counts, row groups, and
confirms the hash column is actually present and sorted where expected.

In [ ]:
metadata_sql = """
SELECT
    file_name,
    count(DISTINCT row_group_id) AS row_groups,
    max(row_group_num_rows) AS max_row_group_rows,
    sum(row_group_compressed_bytes) AS compressed_bytes
FROM parquet_metadata({parquet_glob})
GROUP BY file_name
ORDER BY file_name
"""

runner = BenchmarkRunner(repeats=DEFAULT_REPEATS, progress=False)

for name, glob in DATASETS.items():
    print('\n##', name)
    sql = metadata_sql.format(parquet_glob=sql_literal(glob))
    display(runner.connection.execute(sql).df())

## Run Benchmarks

In [ ]:
@dataclass
class DatasetRun:
    dataset_name: str
    dataset_path: str
    suite: BenchmarkSuite
    params: dict[str, object]
    skipped: dict[str, str]
    results: list[BenchmarkResult]

In [ ]:
def run_dataset(
    runner: BenchmarkRunner, dataset_name: str, dataset_path: str
) -> DatasetRun:
    params, columns = resolve_all_params(runner, dataset_name, dataset_path)
    queries = runnable_queries(SELECTED_QUERIES, columns, params)
    skipped = skipped_queries(SELECTED_QUERIES, columns, params)
    for query_name, reason in skipped.items():
        print(f'{dataset_name} / {query_name}: skipped ({reason})')
    results = []
    for query in queries:
        try:
            result = runner.run_query(query, dataset_path, params)
        except Exception as error:
            print(f'{dataset_name} / {query.name}: {error}')
            continue
        results.append(result)
        print(
            f'{dataset_name} / {result.query}: rows={result.rows} '
            f'best={result.best_seconds:0.4f}s median={result.median_seconds:0.4f}s'
        )
    return DatasetRun(
        dataset_name, dataset_path, BENCHMARK_SUITE, params, skipped, results
    )


def run_matrix(runner: BenchmarkRunner, datasets: dict[str, str]) -> list[DatasetRun]:
    return [
        run_dataset(runner, name, path) for name, path in datasets.items()
    ]


def results_table(dataset_runs: list[DatasetRun]) -> pd.DataFrame:
    rows = [
        {
            'query': result.query,
            'dataset': run.dataset_name,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
        }
        for run in dataset_runs
        for result in run.results
    ]
    if not rows:
        return pd.DataFrame(
            columns=['query', 'dataset', 'rows', 'best_seconds', 'median_seconds']
        )
    return pd.DataFrame(rows).sort_values(['query', 'median_seconds']).reset_index(drop=True)

In [ ]:
runs = run_matrix(runner, DATASETS)
summary = results_table(runs)
display(summary)

### Object-Store Timing

Every result above includes Source Cooperative object-store listing, network
latency, and HTTP range reads. Repeated runs may still benefit from operating
system and DuckDB caches, so use the reported median rather than one timing.

In [ ]:
summary.groupby('dataset')['median_seconds'].describe()

## Analyze Results

The speedup table compares remote median runtimes by query within each dataset
family. Values above 1 mean the hash-sorted remote layout was faster.

In [ ]:
pivot = summary.pivot(index='query', columns='dataset', values='median_seconds')
speedups = pivot.copy()
COMPARISONS = {
    'microsoft_hash_speedup': ('microsoft', 'hashed'),
    'microsoft_hash_matched_speedup': (
        'microsoft', 'hashed_matched_file_count'
    ),
    'cholmes_hash_speedup': ('cholmes', 'cholmes_hashed'),
    'cholmes_hash_4_files_speedup': ('cholmes', 'cholmes_hashed_4_files'),
    'cholmes_hash_8_files_speedup': ('cholmes', 'cholmes_hashed_8_files'),
}
for comparison, (baseline, candidate) in COMPARISONS.items():
    if {baseline, candidate}.issubset(speedups.columns):
        speedups[comparison] = speedups[baseline] / speedups[candidate]

display(speedups.reset_index())

In [ ]:
metadata_rows = []
for dataset, glob in DATASETS.items():
    sql = metadata_sql.format(parquet_glob=sql_literal(glob))
    df = runner.connection.execute(sql).df()
    metadata_rows.append({
        'dataset': dataset,
        'files': len(df),
        'row_groups': int(df['row_groups'].sum()),
        'compressed_gb': float(df['compressed_bytes'].sum() / 1_000_000_000),
        'median_row_groups_per_file': float(df['row_groups'].median()),
    })

file_summary = pd.DataFrame(metadata_rows).sort_values('dataset').reset_index(drop=True)
display(file_summary)

### Reading the Summary

- Compare Microsoft only with `hashed` and `hashed_matched_file_count`; compare
  Holmes only with its one-, four-, and eight-file hashed variants.
- Each raw-versus-hashed comparison includes changes from rewriting compression and
  row-group layout in addition to hash sorting.
- `full_dataset_count` is mostly a metadata/scan control; a big difference here
  usually means compression/schema/row-group overhead, not sorting.
- `composite_stac_search_count` is the main STAC-style query to watch; a hashed win
  here is the strongest signal that sorting helps.
- Hash-filter and hash-order queries are only meaningful on hash-sorted datasets;
  they are reported as skipped for the two raw source datasets.

Use `median_seconds` for comparisons; `best_seconds` is useful for spotting
warm-cache potential but can be optimistic.

## Inspect a Plan

Use this when a timing difference looks interesting. The key line to look
for is `Total Files Read`; if that number drops, DuckDB is pruning files.
Compares the STAC-style search against its hash-range equivalent on the
same hash-sorted dataset.

In [ ]:
dataset_name = 'cholmes_hashed'
run = next(run for run in runs if run.dataset_name == dataset_name)

for query in SELECTED_QUERIES:
    if query.name not in (
        'composite_stac_search_count',
        'composite_hash_stac_search_count',
    ):
        continue
    sql = query.sql.format(
        **{
            key: sql_literal(value)
            for key, value in {'parquet_glob': DATASETS[dataset_name], **run.params}.items()
        }
    )
    print('\n##', query.name)
    print(sql)
    for row in runner.connection.execute('EXPLAIN ANALYZE ' + sql).fetchall():
        print(row[1] if len(row) > 1 else row[0])

## Export Results

Writes one `run.json` per dataset in the same format `cosgp benchmark run`
produces, so any pair can be compared directly with `cosgp benchmark
compare` — including comparisons against a run produced entirely by the
CLI rather than this notebook.

In [ ]:
out_dir = Path('../benchmark-results')
out_dir.mkdir(parents=True, exist_ok=True)

for run in runs:
    run_file = runner.write(
        run.dataset_name,
        run.dataset_path,
        run.suite,
        run.params,
        run.skipped,
        run.results,
        out_dir,
    )
    print(f'wrote {run_file}')

## Results Synthesis

The corrected remote run (`20261027T011639Z`) completed across all seven
datasets. It supports two distinct conclusions. First, rewriting the Microsoft
dataset into smaller, cloud-optimized Parquet files produces large and
repeatable gains. Second, adding the current conservative hash predicate does
not produce a consistent additional speedup. All timings below are median
remote-query times from the corrected run.

### Correctness is restored

| Hashed layout | Ordinary STAC search | Hash-assisted search | Missing |
|---|---:|---:|---:|
| Microsoft, 16 files | 783 | 783 | 0 |
| Microsoft, 12 files | 783 | 783 | 0 |
| Holmes, 1 file | 22 | 22 | 0 |
| Holmes, 4 files | 22 | 22 | 0 |
| Holmes, 8 files | 22 | 22 | 0 |

The previous false negatives came from deriving one interval from only the
spatial-temporal corners. The benchmark now finds conservative hash bounds from
an untimed exact-match scan and retains the exact collection, datetime, and bbox
predicates. This is valid for measuring the timed predicate, but it is not an
end-to-end production strategy because the setup scan is excluded from timing.

### The corrected hash predicate shows no consistent speedup

| Layout | Ordinary search | Hash-assisted | Effect |
|---|---:|---:|---:|
| Microsoft, 16 files | 1.702 s | 1.733 s | 1.8% slower |
| Microsoft, 12 files | 1.807 s | 1.693 s | 6.3% faster |
| Holmes, 1 file | 1.194 s | 1.371 s | 14.8% slower |
| Holmes, 4 files | 1.539 s | 1.879 s | 22.1% slower |
| Holmes, 8 files | 1.479 s | 1.601 s | 8.3% slower |

The mixed Microsoft result and uniformly slower Holmes results do not establish
a practical hash-range benefit. The conservative ranges are also broad: they
contain 9,532 Microsoft rows for 783 exact matches (12.2 candidates per match)
and 726 Holmes rows for 22 exact matches (33 candidates per match). More precise
query-derived ranges would be needed to improve pruning.

### Microsoft rewrite: large and repeatable gains

| Composite query | Raw MS | Rewritten, 16 files | Rewritten, 12 files |
|---|---:|---:|---:|
| STAC search | 15.36 s | 1.70 s — **9.0x** | 1.81 s — **8.5x** |
| STAC + cloud cover | 15.00 s | 1.87 s — **8.0x** | 1.95 s — **7.7x** |
| STAC + geometry | 14.00 s | 3.13 s — **4.5x** | 3.36 s — **4.2x** |
| Datetime-ordered page | 13.80 s | 1.79 s — **7.7x** | 1.78 s — **7.8x** |
| Monthly aggregation | 41.56 s | 2.44 s — **17.0x** | 2.00 s — **20.8x** |
| Latest items | 27.30 s | 2.74 s — **10.0x** | 2.76 s — **9.9x** |
| Scoped ID lookup | 35.92 s | 3.22 s — **11.1x** | 2.62 s — **13.7x** |

The overall Microsoft rewrite is clearly valuable. The two rewritten layouts
are close: 16 files wins the basic, cloud, geometry, and latest-item queries; 12
files wins ordered paging, monthly aggregation, scoped ID lookup, and the
hash-assisted search. These gains cannot be attributed to hashing alone because
the rewrite simultaneously changes compression, size, metadata, row groups,
file count, and sort order.

### Holmes: one rewritten file is the strongest variant

| Composite query | Raw | Rewritten, 1 file | 4 files | 8 files |
|---|---:|---:|---:|---:|
| STAC search | 1.329 s | **1.194 s** | 1.539 s | 1.479 s |
| STAC + cloud | 1.403 s | **1.329 s** | 1.640 s | 1.601 s |
| STAC + geometry | 2.912 s | 1.781 s | 1.805 s | **1.698 s** |
| Ordered page | 1.497 s | 1.640 s | 1.731 s | **1.464 s** |
| Monthly aggregation | **0.603 s** | 0.848 s | 1.171 s | 1.185 s |
| Latest items | **0.955 s** | 1.063 s | 1.307 s | 1.353 s |
| Scoped ID lookup | **0.952 s** | 0.996 s | 1.279 s | 1.304 s |

Holmes is small enough that splitting it usually adds remote-object overhead.
The one-file rewrite modestly improves STAC and cloud searches, while all
rewrites materially improve geometry intersection. Raw remains best for monthly
aggregation, latest items, and scoped ID lookup.

### Bottom line

The strongest result is the value of the complete Microsoft rewrite, not the
hash predicate by itself. The rewritten Microsoft layouts are approximately
4–21x faster across realistic composite queries, while the corrected hash
predicate ranges from 6.3% faster to 22.1% slower. For Holmes, one rewritten
file is the best general rewritten layout, but raw remains best for several
already-cheap queries.

These results should therefore be presented as evidence for cloud-optimized
layout and row-group design. They are not yet evidence that hash sorting or a
hash-range predicate independently improves STAC search performance.

## What This Run Does and Does Not Isolate

The run isolates query behavior across the seven published layouts, and it now
verifies that ordinary and hash-assisted searches return identical counts. It
does **not** isolate hash sorting as a single experimental variable. The raw and
rewritten files differ simultaneously in compression, total size, row-group
count, file count, metadata, and physical ordering.

The hash-assisted timing also excludes the exact scan used to discover safe hash
bounds. That makes it useful for testing the cost of the predicate once bounds
exist, but not for claiming an end-to-end production speedup. A production query
planner must derive conservative ranges from the requested datetime and geometry
without scanning the matching rows first.

## Next Steps

1. **Create a controlled sort-order A/B pair.** Write two remote datasets from
   the same source rows with identical compression, schemas, file counts, file
   sizes, row-group sizes, and metadata. Sort one by the existing baseline order
   and the other by `hash:hash`. This is the comparison needed to isolate the
   value of hash sorting.
2. **Implement query-derived hash ranges.** Replace the benchmark-only exact-scan
   bounds with one or more conservative ranges derived from datetime and bbox.
   Keep the exact STAC predicates and require result-set parity as a correctness
   gate.
3. **Measure pruning directly.** Record files opened, row groups scanned, bytes
   read, candidates admitted by the hash range, and final matches. Wall-clock
   time alone is too sensitive to object-store and cache variation.
4. **Test several query selectivities.** Repeat small, medium, and broad spatial
   and temporal searches. The current Microsoft range admits 9,532 candidates for
   783 matches, while Holmes admits 726 for 22; narrower ranges may matter more
   than changing file count.
5. **Repeat runs before choosing a layout.** Alternate the A/B execution order and
   collect multiple independent remote runs so small differences are not mistaken
   for durable improvements.